# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


# Evidence Notebook 09 - Dense E5 + CSR Final Evaluation

Recovery notebook for frozen Step-9 configuration and results. It does not regenerate CSR representations, embeddings, or retrieval.

In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd
from IPython.display import display, Markdown

CC = Path(r"${THESIS_CONTROL_ROOT}")
WS = Path(r"${THESIS_WORKSPACE}")
SUMMARY_PATH = Path(r"${THESIS_CONTROL_ROOT}\Evidence_Recovery_and_Thesis_Assets") / "06_Validation" / "Notebook_Summaries" / "Evidence_Notebook_09_summary.json"
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest().upper()

print("Evidence notebook mode: frozen-artifact verification only")
print("Correction Control:", CC)
print("Writing Workspace:", WS)


Evidence notebook mode: frozen-artifact verification only
Correction Control: ${THESIS_CONTROL_ROOT}
Writing Workspace: ${THESIS_WORKSPACE}


## Dense E5 + Compact Semantic Representation identity

Frozen order: original text, authoritative roots, compact Maqayis keywords. Query caps are 5 per meaning and 20 overall; passage caps are 3 per meaning and 60 overall.

In [2]:
d = CC / "CSR_Final_Evaluation"
config = json.loads((d / "CSR_CONFIGURATION_FREEZE.json").read_text(encoding="utf-8"))
print(config["system_name"])
display(pd.DataFrame([
 {"Artifact":"Query CSR", **config["inputs"]["query_csr"]},
 {"Artifact":"Passage CSR", **config["inputs"]["passage_csr"]},
 {"Artifact":"Dense+CSR Top100", **config["authoritative_internal_run"]},
]))
assert config["inputs"]["query_csr"]["sha256"] == "6972313CF9A0A8897DA4DD0DF64DB8132A7B68ED766F2C199072ECCBF3D10987"
assert config["inputs"]["passage_csr"]["sha256"] == "7D839476B922628E9123A538DBBDCFF6227D7A762B7866E4A151A8E11964A745"
assert config["authoritative_internal_run"]["sha256"] == "025DAB1CCD7C604888C0834193FC9DB5C5620747CB566B77038823945A959489"
for item in [config["inputs"]["query_csr"],config["inputs"]["passage_csr"],config["authoritative_internal_run"]]:
    assert sha256(Path(item["path"])) == item["sha256"]
print("CSR identities: PASS")

Dense E5 + Compact Semantic Representation (Dense E5 + CSR)


,Artifact,path,sha256,rows,qids
0,Query CSR,${THESIS_CODEX_ROOT}\Master Thesis - ...,6972313CF9A0A8897DA4DD0DF64DB8132A7B68ED766F2C...,251,NaN
1,Passage CSR,${THESIS_CODEX_ROOT}\Master Thesis - ...,7D839476B922628E9123A538DBBDCFF6227D7A762B7866...,1266,NaN
2,Dense+CSR Top100,${THESIS_CODEX_ROOT}\Master Thesis - ...,025DAB1CCD7C604888C0834193FC9DB5C5620747CB566B...,25100,251.0


CSR identities: PASS


## Frozen all-split official and answerable-only results

In [3]:
results = pd.read_csv(d / "CSR_RESULTS_ALL_SPLITS.csv")
display(results[["Split","Population","QID_Count","MAP_at_10","MRR_at_10"]])
test_official = results.query("Split=='test' and Population=='Official judged'").iloc[0]
test_answerable = results.query("Split=='test' and Population=='Answerable-only'").iloc[0]
assert abs(test_official.MAP_at_10 - 0.071858946594) < 1e-12
assert abs(test_official.MRR_at_10 - 0.187519452225) < 1e-12
assert abs(test_answerable.MAP_at_10 - 0.083291051734) < 1e-12
assert abs(test_answerable.MRR_at_10 - 0.217352092352) < 1e-12

,Split,Population,QID_Count,MAP_at_10,MRR_at_10
0,train,Official judged,174,0.137723,0.257460
1,dev,Official judged,25,0.102259,0.259048
2,test,Official judged,51,0.071859,0.187519
3,train,Answerable-only,148,0.161918,0.302689
4,dev,Answerable-only,21,0.121737,0.308390
5,test,Answerable-only,44,0.083291,0.217352


## Query-level Dense E5 comparison

In [4]:
comparison = pd.read_csv(d / "CSR_VS_DENSE_QUERY_COMPARISON.csv")
counts = comparison.Outcome_Group.value_counts().to_dict()
changes = comparison.AP_Change.value_counts().to_dict()
display(pd.DataFrame({"Outcome":counts.keys(),"QIDs":counts.values()}))
display(pd.DataFrame({"Change":changes.keys(),"QIDs":changes.values()}))
assert counts == {"Both fail":19,"Both succeed":16,"Dense-only success":5,"CSR-only success":4}
assert changes == {"unchanged":24,"worsened":15,"improved":5}

,Outcome,QIDs
0,Both fail,19
1,Both succeed,16
2,Dense-only success,5
3,CSR-only success,4


,Change,QIDs
0,unchanged,24
1,worsened,15
2,improved,5


## Final interpretation and thesis references

The preplanned Dense E5 + CSR extension did not improve the Dense E5 baseline overall, although query-level gains occurred for five answerable test qids. Reuse the Step-10 six-system assets for the final comparison.

In [5]:
summary = {
 "query_csr_sha":config["inputs"]["query_csr"]["sha256"],
 "passage_csr_sha":config["inputs"]["passage_csr"]["sha256"],
 "dense_csr_sha":config["authoritative_internal_run"]["sha256"],
 "test_official_map":float(test_official.MAP_at_10),"test_official_mrr":float(test_official.MRR_at_10),
 "test_answerable_map":float(test_answerable.MAP_at_10),"test_answerable_mrr":float(test_answerable.MRR_at_10),
 "csr_only":4,"dense_only":5,"both_success":16,"both_fail":19,
 "improved":5,"worsened":15,"unchanged":24,"substantive_mismatches":0,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Substantive mismatch count: 0")

Substantive mismatch count: 0
